# WM-811K Term Project — Step 1 (v2)
## 환경 설정 + 데이터 로드 + Teacher Baseline
**카테고리 1**: 클래스 불균형 처리 |
**카테고리 2**: Knowledge Distillation
### v2 변경점- **Augmentation on/off 옵션** 추가 (`augment` 인자)- 불균형 처리의 순수 효과를 측정하기 위해 baseline은 **augment=False**로 실행- augmentation은 우리가 선택한 기법(불균형 처리/KD)이 아닌 보조 파이프라인이므로, 통제 변수로 제거 가능---
### 사전 준비로컬에서 만든 `wm811k_64.npz`(39.6MB)를 Drive에 업로드:`MyDrive/wm811k_term/data/wm811k_64.npz`

## 1. GPU 확인

In [20]:
!nvidia-smi

Sun May 31 05:44:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   69C    P0             32W /   72W |    3444MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Google Drive 마운트

In [21]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/wm811k_term'
DATA_PATH = os.path.join(PROJECT_DIR, 'data', 'wm811k_64.npz')
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(DATA_PATH), f'데이터가 없습니다: {DATA_PATH}'
print('데이터 확인됨:', DATA_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
데이터 확인됨: /content/drive/MyDrive/wm811k_term/data/wm811k_64.npz


## 3. 라이브러리 및 시드 설정

In [22]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as tvm
from sklearn.metrics import f1_score
import json

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def get_device():
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

set_seed(42)
device = get_device()
print('Device:', device, '| PyTorch:', torch.__version__)

Device: cuda | PyTorch: 2.11.0+cu128


## 4. 데이터 로드 및 클래스 분포

In [23]:
d = np.load(DATA_PATH, allow_pickle=True)
X_train, y_train = d['X_train'], d['y_train']
X_test, y_test = d['X_test'], d['y_test']
CLASSES = list(d['classes'])
NUM_CLASSES = len(CLASSES)
print(f'Train: {X_train.shape}  Test: {X_test.shape}  Classes: {NUM_CLASSES}\n')
cnts = np.bincount(y_train, minlength=NUM_CLASSES); total = cnts.sum()
print('Train 클래스 분포:')
for i in np.argsort(-cnts):
    print(f'  {CLASSES[i]:<12} {cnts[i]:>8,}  ({cnts[i]/total*100:5.2f}%)')

Train: (138360, 64, 64)  Test: (34590, 64, 64)  Classes: 9

Train 클래스 분포:
  none          117,945  (85.25%)
  Edge-Ring       7,744  ( 5.60%)
  Edge-Loc        4,151  ( 3.00%)
  Center          3,435  ( 2.48%)
  Loc             2,875  ( 2.08%)
  Scratch           954  ( 0.69%)
  Random            693  ( 0.50%)
  Donut             444  ( 0.32%)
  Near-full         119  ( 0.09%)


## 5. Dataset / DataLoader / 불균형 처리 유틸**핵심 변경**: `WaferDataset`에 `augment` 인자 추가. False면 flip을 적용하지 않습니다.

In [24]:
class WaferDataset(Dataset):
    def __init__(self, X, y, train=False, augment=True): # 안쓸거면 False
        self.X, self.y, self.train, self.augment = X, y, train, augment
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        img = self.X[i].astype(np.float32) / 2.0
        if self.train and self.augment:
            if random.random() < 0.5: img = np.fliplr(img).copy()
            if random.random() < 0.5: img = np.flipud(img).copy()
        return torch.from_numpy(img).unsqueeze(0), int(self.y[i])

def compute_class_weights(y, num_classes=NUM_CLASSES):
    counts = np.bincount(y, minlength=num_classes).astype(np.float64)
    counts[counts == 0] = 1.0
    w = 1.0 / counts
    return torch.tensor(w / w.sum() * num_classes, dtype=torch.float32)

def make_loaders(X_tr, y_tr, X_te, y_te, batch_size=128, oversample=False, augment=True):
    tr_ds = WaferDataset(X_tr, y_tr, train=True, augment=augment)
    te_ds = WaferDataset(X_te, y_te, train=False, augment=False)
    if oversample:
        counts = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float64)
        counts[counts == 0] = 1.0
        sample_w = (1.0 / counts)[y_tr]
        sampler = WeightedRandomSampler(
            torch.tensor(sample_w, dtype=torch.double), len(y_tr), replacement=True)
        tr = DataLoader(tr_ds, batch_size=batch_size, sampler=sampler, num_workers=8, pin_memory=True)
    else:
        tr = DataLoader(tr_ds, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True)
    te = DataLoader(te_ds, batch_size=256, shuffle=False, num_workers=8, pin_memory=True)
    return tr, te

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__(); self.gamma = gamma; self.alpha = alpha
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1); p = torch.exp(logp)
        logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
        loss = -((1 - p_t) ** self.gamma) * logp_t
        if self.alpha is not None:
            loss = self.alpha.to(logits.device)[target] * loss
        return loss.mean()

print('데이터 유틸 정의 완료 (augment 옵션 포함)')

데이터 유틸 정의 완료 (augment 옵션 포함)


In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 6. 모델 정의

In [26]:
def get_teacher(num_classes=NUM_CLASSES):
    m = tvm.resnet18(weights=None)
    m.conv1 = nn.Conv2d(1, 64, 3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

class StudentCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(1,32), block(32,64), block(64,128), block(128,256))
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(256, num_classes)
    def forward(self, x):
        x = self.features(x); x = self.gap(x).flatten(1); return self.classifier(x)

def get_student(num_classes=NUM_CLASSES):
    return StudentCNN(num_classes)

def get_mobilenet(num_classes=NUM_CLASSES):
    m = tvm.mobilenet_v2(weights=None)
    old = m.features[0][0]
    m.features[0][0] = nn.Conv2d(1, old.out_channels, 3, stride=2, padding=1, bias=False)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)
    return m

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

for name, fn in [('Teacher (ResNet-18)', get_teacher), ('Student (CNN)', get_student), ('MobileNetV2', get_mobilenet)]:
    n = count_params(fn()); print(f'{name:<22} {n:>12,}  ({n/1e6:.2f}M)')
t, s = count_params(get_teacher()), count_params(get_student())
print(f'\n압축비 Teacher/Student: {t/s:.1f}x')

Teacher (ResNet-18)      11,172,297  (11.17M)
Student (CNN)               390,633  (0.39M)
MobileNetV2               2,234,825  (2.23M)

압축비 Teacher/Student: 28.6x


## 7. 학습 / 평가 함수

In [27]:
def train_one_epoch(model, loader, optimizer, criterion, use_amp=True):
    model.train(); total=correct=0; loss_sum=0.0
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(x); loss = criterion(out, y)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()*x.size(0); correct += (out.argmax(1)==y).sum().item(); total += x.size(0)
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); preds=[]; trues=[]
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        preds.append(model(x).argmax(1).cpu().numpy()); trues.append(y.numpy())
    pred, true = np.concatenate(preds), np.concatenate(trues)
    acc = (pred==true).mean()
    macro_f1 = f1_score(true, pred, average='macro', zero_division=0)
    per_class = f1_score(true, pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
    return {'accuracy':float(acc),'macro_f1':float(macro_f1),'per_class_f1':per_class.tolist()}

def print_per_class(per_class_f1):
    print('  Per-class F1:')
    for c, f in zip(CLASSES, per_class_f1): print(f'    {c:<12} {f:.4f}')
print('학습/평가 함수 정의 완료')

학습/평가 함수 정의 완료


## 8. 공용 학습 함수`imbalance`와 `augment`를 인자로 받아 baseline / 불균형 ablation을 모두 처리합니다.- `imbalance`: none / oversample / focal / classweight- `augment`: True / False (baseline 비교는 False)체크포인트와 결과 JSON은 `{tag}` 이름으로 Drive에 저장됩니다.

In [28]:
def run_teacher(imbalance='none', augment=False, epochs=30, batch=128, lr=0.1, tag=None):
    set_seed(42)
    if tag is None:
        tag = f"{imbalance}_aug{int(augment)}"
    oversample = (imbalance == 'oversample')
    tr_loader, te_loader = make_loaders(X_train, y_train, X_test, y_test, batch, oversample, augment)

    if imbalance == 'focal':
        criterion = FocalLoss(gamma=2.0)
    elif imbalance == 'classweight':
        criterion = nn.CrossEntropyLoss(weight=compute_class_weights(y_train).to(device))
    else:
        criterion = nn.CrossEntropyLoss()

    model = get_teacher().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    ckpt_path = os.path.join(CKPT_DIR, f'teacher_{tag}.pt')

    print(f'>>> 학습 시작: imbalance={imbalance}, augment={augment}, epochs={epochs}, tag={tag}\n')
    history, best_f1 = [], 0.0
    for ep in range(1, epochs+1):
        tr_loss, tr_acc = train_one_epoch(model, tr_loader, optimizer, criterion)
        scheduler.step()
        m = evaluate(model, te_loader)
        history.append({'epoch':ep,'train_loss':tr_loss,'test_acc':m['accuracy'],
                        'macro_f1':m['macro_f1'],'per_class_f1':m['per_class_f1']})
        print(f"Epoch {ep:02d} | loss {tr_loss:.3f} | test_acc {m['accuracy']:.4f} | macro_f1 {m['macro_f1']:.4f}")
        if m['macro_f1'] > best_f1:
            best_f1 = m['macro_f1']; torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    final = evaluate(model, te_loader)
    print(f"\n{'='*50}\n[{tag}] 최종 (best macro-F1 모델)")
    print(f"  Accuracy : {final['accuracy']*100:.2f}%\n  Macro-F1 : {final['macro_f1']:.4f}")
    print_per_class(final['per_class_f1'])
    with open(os.path.join(RESULTS_DIR, f'teacher_{tag}.json'), 'w') as f:
        json.dump({'imbalance':imbalance,'augment':augment,'accuracy':final['accuracy'],
                   'macro_f1':final['macro_f1'],'per_class_f1':final['per_class_f1'],
                   'classes':CLASSES,'history':history}, f, indent=2)
    print(f"\n저장 완료 → teacher_{tag}.pt / teacher_{tag}.json")
    return final

print('run_teacher 정의 완료')

run_teacher 정의 완료


## 9. Teacher Baseline 학습 (augmentation 없음, 불균형 처리 없음)이것이 카테고리 1 ablation의 기준점입니다. 30 epoch.**관전 포인트**: augmentation을 끈 baseline의 macro-F1이 이전 결과(aug 적용 시 0.90)보다얼마나 낮아지는지 확인하세요. 낮아질수록 불균형 처리로 향상시킬 여지가 큽니다.

In [29]:
import os
print("CPU 코어:", os.cpu_count())

CPU 코어: 12


In [30]:
baseline = run_teacher(imbalance='none', augment=False, epochs=30)

>>> 학습 시작: imbalance=none, augment=False, epochs=30, tag=none_aug0

Epoch 01 | loss 0.286 | test_acc 0.9170 | macro_f1 0.3240
Epoch 02 | loss 0.141 | test_acc 0.9416 | macro_f1 0.4995
Epoch 03 | loss 0.121 | test_acc 0.9535 | macro_f1 0.6740
Epoch 04 | loss 0.116 | test_acc 0.9580 | macro_f1 0.5838
Epoch 05 | loss 0.110 | test_acc 0.9580 | macro_f1 0.7279
Epoch 06 | loss 0.105 | test_acc 0.9606 | macro_f1 0.7604
Epoch 07 | loss 0.101 | test_acc 0.9589 | macro_f1 0.7541
Epoch 08 | loss 0.097 | test_acc 0.9401 | macro_f1 0.6807
Epoch 09 | loss 0.095 | test_acc 0.9456 | macro_f1 0.6683
Epoch 10 | loss 0.091 | test_acc 0.9628 | macro_f1 0.7315
Epoch 11 | loss 0.087 | test_acc 0.9690 | macro_f1 0.7923
Epoch 12 | loss 0.086 | test_acc 0.9600 | macro_f1 0.7267
Epoch 13 | loss 0.083 | test_acc 0.9628 | macro_f1 0.7865
Epoch 14 | loss 0.080 | test_acc 0.9667 | macro_f1 0.8142
Epoch 15 | loss 0.077 | test_acc 0.9694 | macro_f1 0.8358
Epoch 16 | loss 0.073 | test_acc 0.9574 | macro_f1 0.7665
Epoc

## 10. (참고) augmentation 적용 baseline선택 사항: augmentation의 효과를 보여주는 비교용. 로컬에서 이미 macro-F1 0.90을 확인했으므로시간을 아끼려면 건너뛰고 로컬 결과를 보고서에 인용해도 됩니다.

In [ ]:
# 필요 시 주석 해제하여 실행 (비교용)
# baseline_aug = run_teacher(imbalance='none', augment=True, epochs=30)

---### Step 1 완료 후- **augment=False baseline**의 macro-F1과 소수 클래스 F1을 확인하세요.- 이 값이 카테고리 1(불균형 처리) ablation의 기준점이 됩니다.- 다음 단계에서 `run_teacher('oversample', augment=False)` 등으로 비교합니다.